# DL Assignment 02 — Real2Sim: 휴대폰 영상으로 천의 공기 저항과 굽힘 강성 찾기 (메쉬부터 끝까지 · 코드 설명판)

**천 사진 한 장 + 자로 잰 가로 · 세로**로 시뮬레이터 메쉬를 만들고, 여러분이 찍은 영상 한 편으로
① **공기 저항**을 찾고 ② **굽힘 강성의 상한**을 재서 ③ 그 아래에서 강성 값을 정합니다.
여러분이 채울 것은 **TODO 함수 7개**(③ 3개, ⑤ 4개)와 `TODO` 주석이 붙은 설정입니다. 나머지 코드는 주어진 것이고, 설명은 그 코드가 무엇을 하는지 이해하기 위한 것입니다.
셀을 잘게 나누어 각 단계 앞에 **무엇을 · 왜 · 입력과 출력의 모양(shape) · 라이브러리 함수 안에서 일어나는 일 · 결과에서 볼 것**을 적었습니다.

**목표.** 천(여러분이 잰 `WIDTH_CM` × `HEIGHT_CM`, 질량 `CLOTH_MASS_G`)을 두 손으로 들고 움직이는 RGB 영상 한 편만 보고,
MuJoCo 속 천이 영상과 같이 움직이도록 시뮬레이터의 물성 두 개를 정합니다. depth 카메라는 쓰지 않습니다.

| 물성 | 영상으로 알 수 있는 것 | 이 노트북의 결과 |
|---|---|---|
| 공기 저항 `air` | **값 하나** (손실이 log(air)에 대해 U자) | `air` (N·s/m³, 천 1 m²당) |
| 굽힘 강성 `young` | **상한만** (그보다 부드러운 값은 영상으로 구분 안 됨) | `young_upper`, 그리고 그 아래에서 고른 `young` |

**왜 강성은 상한만인가 (수업에서 잰 것).** 매달린 천의 모양을 강성이 바꾸는 범위는 "굽힘 길이" (보통 천 1–4 cm) 정도입니다.
천(30 cm 이상)이 그보다 훨씬 크면 모양은 중력과 손 위치로 거의 정해지고, 강성은 모서리와 작은 주름만 바꿉니다.
그래서 부드러운 쪽(1e2–1e4)은 모두 영상과 똑같이 잘 맞고, **너무 뻣뻣한 값만 영상과 달라 보입니다.**
교수님 영상 T01: 공기 저항 0.6 근처에서 손실이 뚜렷이 최소, 강성은 1e2–3e4가 모두 같고 1e5부터 나빠짐.

**순서**: **천 사진 → 메쉬(M0)** → 촬영과 보정(M1) → 인식(M2) → 3-D로(M3, TODO 3개) → **⑤a 공기 저항 → ⑤b 강성 상한 → ⑤c 강성 값**(M4, TODO 4개)
→ 합성 영상 검증(M5) → `results_<팀>.zip` 제출. 바꿔야 하는 값은 `TODO` 주석이 붙은 설정 셀뿐입니다.

```
사진 + 자 ─▶ ⓪ 메쉬 (WIDTH × HEIGHT)
영상 ─▶ ① 카메라 K ─▶ ② 인식 ─────────────────────▶ ③ 3-D로 ─────▶ ④ 시뮬레이터 ─▶ ⑤a air ─▶ ⑤b young 상한 ─▶ ⑤c young ─▶ ⑥ 검증
        (체커보드)     DINO: 글→상자                    scale,          두 손이 천을     (U자 곡선의    (평평 → 오르는     (상한 아래    합성 영상
                       SAM 2: 상자→마스크→모서리        세로선으로 중력  움직임 + 공기   최솟값)         지점)              log 중간)    (정답 앎)
                       CoTracker3: 격자점 121개
                       Depth Anything V2: 깊이 (m, 부정확)
```

### 이 노트북에 나오는 주요 변수

| 이름 | 모양 | 뜻 |
|---|---|---|
| `WIDTH_CM`, `HEIGHT_CM` | 숫자 | 자로 잰 천의 두 변 (cm): c0→c1, c0→c3 |
| `mesh` | dict | 시뮬레이터 메쉬: `V` 꼭짓점 (N, 3) m, `F` 삼각형, `uv`, `corners`, `width`, `height` |
| `frames` | (T, H, W, 3) uint8 | 영상 프레임 (RGB, 폭 960으로 줄임) |
| `t` | (T,) | 각 프레임의 시각 (s). 휴대폰은 fps가 일정하지 않아 파일의 시각을 씀 |
| `K` | (3, 3) | 카메라 내부 파라미터 [[fx, 0, cx], [0, fy, cy], [0, 0, 1]] |
| `box` | (4,) | DINO가 찾은 상자 (x0, y0, x1, y1) 픽셀 |
| `mask_reg` | (H, W) bool | 등록 프레임의 천 마스크 |
| `C` | (4, 2) | 천의 모서리 c0, c1, c2, c3 (픽셀) |
| `uv` | (P, 2), P = 121 | 추적할 점이 천의 어디인지 (0–1 texture 좌표) |
| `xy`, `vis` | (T', P, 2), (T', P) | 추적된 픽셀 위치, 보이는지. T' = END − START + 1 |
| `depth` | (T', H/2, W/2) | 프레임마다 깊이 지도 (m) |
| `obs` | dict | 3-D 관찰: 천의 점 `P` (T', P, 3), 두 손 `hands` (T', 2, 3), 카메라 `R`, `tw` … |
| `up` | (3,) | 카메라 좌표에서 "위" 방향 (세로선 또는 바닥 평면) |
| `A` | dict | ⑤a 결과: `air`, 공기 저항마다의 손실 `curve` |
| `Y` | dict | ⑤b–c 결과: `young_upper` (상한), `young` (고른 값), 허용치 `tol_mm`, young마다의 손실 `curve` |

**런타임 → 런타임 유형 변경 → T4 GPU**를 고른 뒤 위에서부터 차례로 실행하세요.

## 0. 준비

**무엇을.** 수업 저장소를 `/content/repo`에 받고, Colab에 없는 패키지를 설치하고, 저장소의 모듈을 import 합니다.

- 저장소 주소(`REPO_URL`)가 아직 자리표시자(`<org>`)이면 **zip 업로드**를 받습니다 → 수업 저장소를 zip으로 받아 올리세요.
- `TEAM`을 여러분의 팀 ID로 바꾸세요.
- `mujoco==4.0.1`: 천의 **굽힘 탄성 + 늘어나지 않음**을 같이 쓰려면 4.x가 필요합니다 (3.3.1은 두 가지를 같이 쓰면 천이 20–57 % 늘어나거나 발산).
- `MUJOCO_GL=egl`: 화면 없는 Colab에서 MuJoCo가 GPU로 그림을 그리게 합니다.

| 모듈 | 하는 일 |
|---|---|
| `r2s/photo2mesh_wh.py` | ⓪ 천 사진 + 가로 · 세로 → 메쉬 (이 노트북에서 처음 씀) |
| `r2s/take.py` | 영상 읽기, 체커보드 보정, 마스크 → 모서리 |
| `r2s/perceive.py` | 사전학습 모델 4개 (DINO, SAM 2.1, CoTracker3, Depth Anything V2) |
| `r2s/lift.py` | ③ 2-D 추적 + 깊이 → 3-D, 세로선으로 중력 (TODO 3개) |
| `sim/cloth_sim.py` | ④ MuJoCo 천 시뮬레이터 (굽힘 강성 `young` + 공기 저항 `air`) |
| `r2s/fit.py` | ⑤ 공기 저항 → 강성 상한 → 강성 값 (TODO 4개) |
| `r2s/check.py`, `r2s/synth.py`, `r2s/viz.py` | TODO 테스트, 합성 영상, 그림 |

In [ ]:
import os, sys, json, subprocess, time
TEAM = 'T01'                                                           # TODO: 팀 ID (결과 폴더 이름 results_<TEAM>)
REPO_URL = 'https://github.com/<org>/DL_assignment02.git'           # 수업 저장소 (자리표시자면 zip 업로드)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('/content/repo'):                            # 다시 실행해도 한 번만 받음
        if '<org>' in REPO_URL:
            from google.colab import files; print('저장소 zip을 올리세요'); up = files.upload(); z = next(iter(up))
            subprocess.run(['unzip', '-q', z, '-d', '/content/_r'], check=True); top = [p for p in os.listdir('/content/_r')][0]; os.rename(f'/content/_r/{top}', '/content/repo')
        else: subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/repo'], check=True)
    os.chdir('/content/repo'); os.environ['MUJOCO_GL'] = 'egl'          # 화면 없이 GPU로 렌더링
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mujoco==4.0.1', 'transformers>=4.56', 'accelerate', 'imageio-ffmpeg'], check=True)
else:
    os.chdir('..' if os.path.basename(os.getcwd()) in ('notebooks', 'grading') else '.')
sys.path[:0] = [os.path.abspath('sim'), os.path.abspath('r2s')]       # 저장소의 sim/, r2s/ 를 import 경로에
import numpy as np, cv2, torch, mujoco
import take, perceive, lift, fit, viz, check, synth
from cloth_sim import ClothSim, load_mesh, at_uv
from IPython.display import Video, display
import shutil, matplotlib.pyplot as plt
OUT = f'results_{TEAM}'; os.makedirs(OUT, exist_ok=True)               # 모든 결과(그림, 영상, npz, json)가 여기에
print('mujoco', mujoco.__version__, '| device', perceive.DEV)

## ⓪ 천 사진 → 시뮬레이터 메쉬 (M0)

**무엇을.** 바닥이나 책상에 **평평하게 펼친 천의 사진 한 장**과 자로 잰 **두 변의 길이**로, MuJoCo가 읽는 삼각형 메쉬를 만듭니다
(`cloth_mesh.npz`: 약 2 cm 간격의 격자, 실제 크기 m, z = 0 평면, 가운데가 원점).

**왜 중요한가.** 천의 크기는 이후 모든 단계에 들어갑니다.
- ③ 깊이 scale: 이웃 격자점 사이의 **실제 거리** = uv 차이 × 천의 가로 · 세로. 크기가 10 % 틀리면 scale이 10 % 틀리고, 손 궤적과 바닥 평면도 같이 틀립니다.
- ④ 시뮬레이터: 시뮬레이션 천이 이 크기, 이 격자로 만들어집니다. 크기가 다르면 같은 손 움직임에서 처지는 모양이 달라지고, 그 차이를 `young`이 대신 메우려 합니다.
- 교수님 휴대폰 영상(results_take_A)에서는 **다른 천의 사진으로 만든 메쉬**를 써서, 강성이 정해지지 않은 원인 중 하나가 되었습니다.

**두 변의 이름** (모서리 이름은 ②와 같음):

```
   c0 ●━━━━━━━━━━ HEIGHT_CM: 두 손이 잡는 변 ━━━━━━━━━━● c3
      ┃                                                  ┃
   WIDTH_CM                                           WIDTH_CM
      ┃                                                  ┃
   c1 ●━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━● c2

   메쉬:  x 방향 = c0 → c1 = WIDTH,   y 방향 = c0 → c3 = HEIGHT     (②의 uv: u = c0→c1, v = c0→c3)
```

`WIDTH_CM`는 손이 잡지 않는 변 (들면 아래로 늘어지는 방향). 외우는 법: **HEIGHT = 두 손 사이**. 영상에서 어느 변을 잡을지 먼저 정하고, 그 변을 `HEIGHT_CM`으로 재세요.
사진에서 천이 어느 방향으로 놓였는지는 상관없습니다 (스크립트가 사진 속 비율로 어느 변이 어느 쪽인지 정함). 단, 정사각형에 가까우면 구분이 어려우니 아래 그림에서 c0–c3를 꼭 확인하세요.

**사진 찍는 법**
- 천을 **주름 없이 평평하게**, 천과 색이 다른 바닥이나 책상 위에. 네 모서리가 **모두** 보이게.
- 위에서 내려다보거나 조금 비스듬히 (두 변의 길이를 다 알기 때문에 비스듬해도 됩니다). 줌 1×, 그림자 · 손 · 다른 물건이 천 위에 없게.
- JPG 또는 PNG (iPhone HEIC는 JPG로 바꿔서). 원본 해상도 그대로 올리면 됩니다.

**재는 법**: 줄자로 두 변을 각각 **2–3군데** 재서 평균 (가장자리 끝에서 끝까지, mm 단위까지). 주방 저울이 있으면 질량도 재세요 (얇은 면 천 30 × 30 cm ≈ 20 g).

In [ ]:
# ===== 설정: 자로 잰 천 =====
WIDTH_CM     = 27.8    # TODO: c0 -> c1, 손이 잡지 않는 변 (cm)
HEIGHT_CM    = 32.0    # TODO: c0 -> c3, 두 손이 잡는 변 (cm)
CLOTH_MASS_G = 20.0    # TODO: 천 질량 (g). 저울이 없으면 20
MESH = 'assets/my_cloth'       # 새 메쉬를 저장할 폴더
SEG = 'sam'            # 'sam': Grounding DINO + SAM 2.1 (권장) | 'grabcut': OpenCV (딥러닝 없음) | 'corners': CORNERS를 직접
MESH_PROMPT = 'cloth.' # SEG='sam'일 때 DINO에 줄 글 (천이 잘 안 잡히면 'towel.', 'white cloth.' 등)
CORNERS = None         # SEG='corners'일 때: "x,y;x,y;x,y;x,y" 원본 사진의 모서리 픽셀 (순서 상관없음)

In [ ]:
if IN_COLAB:
    from google.colab import files
    print('평평하게 펼친 천의 사진(JPG/PNG)을 올리세요'); MESH_PHOTO = next(iter(files.upload()))
else:
    MESH_PHOTO = 'data/my_cloth.jpg'                                   # 로컬에서 돌릴 때
print('photo:', MESH_PHOTO, '-> mesh folder:', MESH)

### ⓪-1 `photo2mesh_wh.py` 안에서 일어나는 일

| 단계 | 처리 | 이유 |
|---|---|---|
| 1 분할 | Grounding DINO(`MESH_PROMPT` → 상자) → SAM 2.1(상자 → 마스크). ②와 같은 두 모델 | 천 영역만 |
| 2 사각형 | 마스크 외곽선 → 볼록 껍질 → 꼭짓점 4개 → **각 변의 가운데 70 % 점에 직선을 robust fit** → 직선끼리의 교점 = 모서리 | 모서리 부근의 실밥 · 말림에 덜 흔들림 |
| 3 초점거리 f | 사진 속 사각형이 "직각이고 비율이 WIDTH : HEIGHT인 직사각형"이 되는 f를 1-D 탐색 (log 격자 → 황금분할) | EXIF 없이 카메라를 구함 |
| 4 라벨 | 사진의 어느 변을 c0→c1로 볼지 두 가지 중, 사진 속 비율이 잰 비율과 맞는 쪽 | 천을 어느 방향으로 놓고 찍어도 됨 |
| 5 자세 | `cv2.solvePnP` (IPPE, 평면용): 실제 크기 직사각형의 네 모서리 ↔ 사진의 픽셀 → 천 평면의 R, t | 사진을 천 평면으로 펴기 위해 |
| 6 texture | 사진을 천 평면 위로 펴서 `texture.png` (2 px/mm, 정면에서 본 무늬) | 시각화 · 확인용 |
| 7 격자 | 간격 2 cm에 가장 가까운 nx × ny 격자, 삼각형 대각선을 번갈아, 꼭짓점마다 uv | 시뮬레이터가 2 cm 격자에 맞춰져 있음 |
| 8 확인 | 메쉬를 맞춘 카메라로 사진 위에 다시 그려 **마스크와의 IoU**, 사진 속 비율과 잰 비율의 **차이(%)** | 모서리 · 크기가 맞는지 |

**3단계의 식.** 단위 정사각형 uv ∈ [0, 1]² → 사진 픽셀의 homography를 $H$라 하면, 핀홀 카메라에서
$$K^{-1}H = \lambda\,[\,W\,\mathbf r_1 \;\; H_c\,\mathbf r_2 \;\; \mathbf t\,] \quad(\mathbf r_1 \perp \mathbf r_2,\ |\mathbf r_1| = |\mathbf r_2| = 1)$$
이므로 $K^{-1}H$의 앞 두 열 $\mathbf b_1, \mathbf b_2$는 **직교**하고 길이 비가 $|\mathbf b_1| / |\mathbf b_2| = W / H_c$ (잰 가로 / 세로) 여야 합니다.
주점은 사진 중심으로 두고, 두 조건의 오차 $\cos^2(\mathbf b_1, \mathbf b_2) + (\log\frac{|\mathbf b_1|}{|\mathbf b_2|} - \log\frac{W}{H_c})^2$를 최소로 하는 $f$를 찾습니다.
긴 변 하나만 재는 `photo2mesh.py`(`--side_cm`)는 직각 조건만 쓰고 짧은 변은 사진에서 계산하므로, 사진을 거의 정면에서 찍으면 f가 정해지지 않고 짧은 변이 틀리기 쉽습니다.

**PASSED 조건**: IoU > 0.85, 35 mm 환산 초점거리 15–60 mm (휴대폰 기본 카메라는 23–28 mm), 사진 속 비율과 잰 비율의 차이 ≤ 5 %.

In [ ]:
assert os.path.exists('r2s/photo2mesh_wh.py'), '저장소에 r2s/photo2mesh_wh.py가 없습니다: 새 저장소 zip을 받으세요'
cmd = [sys.executable, 'r2s/photo2mesh_wh.py', '--photo', MESH_PHOTO, '--out', MESH,
       '--width_cm', str(WIDTH_CM), '--height_cm', str(HEIGHT_CM), '--seg', SEG, '--prompt', MESH_PROMPT]
if SEG == 'corners': cmd += ['--corners', CORNERS]
t0 = time.time(); r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout); print(f'({time.time() - t0:.0f} s)')
if r.returncode: print(r.stderr[-3000:]); raise RuntimeError('photo2mesh_wh failed (위 메시지)')
shutil.copy(MESH_PHOTO, os.path.join(MESH, 'photo' + os.path.splitext(MESH_PHOTO)[1].lower()))   # 원본 사진도 같이 보관

In [ ]:
mrep = json.load(open(f'{MESH}/mesh_report.json', encoding='utf-8'))
rd = lambda n: cv2.cvtColor(cv2.imdecode(np.fromfile(f'{MESH}/{n}', np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
viz.show(viz.sheet([rd('mask_overlay.jpg'), rd('corners.jpg'), rd('reprojection.jpg')], cols=3, w=560,
                   labels=['1 mask', '2 corners c0..c3', '8 mesh drawn back']), 'photo -> mesh checks', (18, 5))
print({k: mrep.get(k) for k in ('width_cm', 'height_cm', 'photo_aspect', 'aspect', 'aspect_diff_pct', 'corners_relabelled',
                                'focal_px', 'focal_35mm', 'iou_mesh_vs_mask', 'grid', 'n_vertices', 'passed')})

> **확인.**
> - **1 mask**: 빨간 영역이 천 전체이고 바닥 · 그림자 · 손을 포함하지 않는지. 틀리면 `MESH_PROMPT`를 바꾸거나 `SEG = 'grabcut'`, 그래도 안 되면 `SEG = 'corners'`와 `CORNERS`.
> - **2 corners**: 초록 사각형이 천의 네 변에 붙어 있는지, 그리고 **c0 → c3가 손으로 잡을 변**(`HEIGHT_CM`)인지. 스크립트는 c0 → c1이 `WIDTH_CM`가 되도록 라벨을 붙입니다 (`corners_relabelled = True`이면 사진의 방향을 돌려서 맞춘 것).
> - **`aspect_diff_pct`**: 사진에서 본 가로 / 세로 비율과 자로 잰 비율의 차이. **±5 % 안**이어야 합니다. 크면 (a) 모서리 하나가 빗나갔거나 (2번 그림), (b) 천이 평평하지 않았거나, (c) 두 변을 바꿔 넣었거나 잘못 잰 것입니다.
> - **8 mesh drawn back**: 메쉬를 맞춘 카메라로 사진에 그린 것이 천과 겹치는지 (`iou_mesh_vs_mask` > 0.85).
> - `focal_35mm`가 15–60 mm 밖이면 사진이 너무 정면이거나 모서리가 틀린 것입니다.

### ⓪-2 메쉬 읽기와 천 질량

`load_mesh(MESH)`가 돌려주는 dict: `V` 꼭짓점 (N, 3) m, `F` 삼각형 (M, 3), `uv` 꼭짓점마다 texture 좌표, `corners` 네 모서리 꼭짓점 번호 (c0, c1, c2, c3), `width` (c0→c1, m), `height` (c0→c3, m).

**질량.** `sim/cloth_sim.py`의 `cloth_sim.MASS` 기본값은 20 g입니다. 아래 셀은 그것을 `CLOTH_MASS_G`로 바꿉니다.
`fit`(병렬 시뮬레이션에도 이 값을 넘김)과 `synth.make_synth`가 모두 이 값을 쓰므로 ④–⑥ 전체에 적용됩니다.
(굽힘 강성과 질량은 함께 모양을 정합니다: 처짐은 대략 **굽힘 강성 / 무게**의 비로 정해지므로, 질량이 2배 틀리면 강성 상한도 약 2배 틀립니다.
공기 저항도 마찬가지로 **공기 저항 / 질량**이 운동이 줄어드는 빠르기를 정합니다.)

In [ ]:
mesh = load_mesh(MESH)
import cloth_sim; cloth_sim.MASS = CLOTH_MASS_G / 1000                 # 모든 시뮬레이션(④–⑥, 병렬 실행 포함)의 천 질량
print(f'cloth {mesh["width"] * 100:.1f} (c0->c1) x {mesh["height"] * 100:.1f} (c0->c3) cm, mass {ClothSim(MESH).mass * 1000:.1f} g')
print('mesh:', {k: getattr(v, 'shape', v) for k, v in mesh.items() if k in ('V', 'F', 'uv', 'corners')})

V, F = mesh['V'], mesh['F']
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
ax[0].triplot(V[:, 0] * 100, V[:, 1] * 100, F, lw=.5, color='#1971C2'); ax[0].set_aspect('equal')
for i, vid in enumerate(mesh['corners']):
    ax[0].plot(*V[vid, :2] * 100, 'o', color='#E8590C'); ax[0].annotate(f'c{i}', V[vid, :2] * 100, xytext=(6, 6), textcoords='offset points', color='#E8590C', fontsize=13)
ax[0].plot(*(V[mesh['corners'][[0, 3]], :2] * 100).T, color='#2F9E44', lw=4, label='held side (c0-c3)')
ax[0].set_xlabel('x (cm): c0 -> c1 = width'); ax[0].set_ylabel('y (cm): c0 -> c3 = height'); ax[0].legend(loc='lower right'); ax[0].set_title(f'simulator mesh: {len(V)} vertices, {len(F)} triangles')
if os.path.exists(f'{MESH}/texture.png'): ax[1].imshow(rd('texture.png')); ax[1].set_title('texture.png (the photo, flattened onto the cloth)')
ax[1].axis('off'); plt.tight_layout(); plt.show()

> **확인.** 크기가 자로 잰 값과 같은지, 초록 변(c0–c3)이 손으로 잡을 변인지 보세요. 격자 간격은 2 cm에 가깝게 정해지므로 (예: 27.8 cm → 15칸이면 1.99 cm), 꼭짓점 수는 천 크기에 따라 달라집니다 (27.8 × 32 cm: 255개).

## 1. 영상과 카메라 (M1)

**무엇을.** 천 영상과, 카메라 내부 파라미터 **K**를 구할 자료(체커보드 영상 또는 `camera.json`)를 올립니다.
[docs/FILMING.md](../docs/FILMING.md)대로 **같은 휴대폰, 같은 설정**(해상도, 줌 1×, 손떨림 보정 끔)으로 찍은 천 영상과 체커보드 영상을 올리세요.
설정이 다르면 K가 달라집니다. 이미 보정해서 `camera.json`(fx, fy, cx, cy, width)이 있으면 체커보드 영상 대신 그것을 올려도 됩니다.

**왜 K가 필요한가.** 깊이 모델은 픽셀마다 거리 z만 줍니다. 그 점이 옆으로 몇 m인지는 핀홀 모델로 계산합니다:

$$x = \frac{u - c_x}{f_x}\,z,\qquad y = \frac{v - c_y}{f_y}\,z$$

반대로 ⑤에서 시뮬레이션 천을 영상 위에 그릴 때도 K로 투영합니다. fx가 틀리면 옆 방향만 틀리고 깊이 방향은 그대로라서,
③의 scale 보정으로도 고칠 수 없는 **형상 왜곡**(깊이 방향으로 늘어나거나 줄어듦)과 중력 방향 오차가 생깁니다.

In [ ]:
if IN_COLAB:
    from google.colab import files
    print('1) 천 영상을 올리세요'); CLOTH_VIDEO = next(iter(files.upload()))
    print('2) 체커보드 영상을 올리세요 (없으면 취소). 카메라 파라미터 파일(camera.json: fx fy cx cy width)이 이미 있으면 그것을 올려도 됩니다')
    try: BOARD_VIDEO = next(iter(files.upload()))
    except Exception: BOARD_VIDEO = None
else:
    CLOTH_VIDEO, BOARD_VIDEO = 'data/my_take.mp4', 'data/my_board.mp4'          # 로컬에서 돌릴 때

### 1-1 영상 읽기
`take.load_video(path, width=960)`은 모든 프레임을 RGB로 읽어 **폭 960**으로 줄이고, 각 프레임의 **시각** `t`를 파일에서 읽습니다.
휴대폰 영상은 fps가 조금씩 변하므로 `프레임 번호 / fps` 대신 실제 시각을 씁니다 (시뮬레이터의 손이 그 시각에 맞춰 움직임).

In [ ]:
frames, t = take.load_video(CLOTH_VIDEO, width=960); T, H, W = frames.shape[:3]
print(f'{T} frames, {t[-1]:.1f} s ({T / t[-1]:.1f} fps), {W}x{H}')
print('frames', frames.shape, frames.dtype, '| t', t.shape, f'first steps {np.diff(t[:4]).round(4)}')

### 1-2 카메라 내부 파라미터 K
세 가지 경우를 처리합니다.

1. **`camera.json`**: 원래 해상도 `width`에서의 fx, fy, cx, cy를 읽어, 영상을 줄인 비율 `k = W / width`만큼 곱합니다.
   (영상을 절반으로 줄이면 같은 물체가 절반 픽셀로 보이므로 fx, cx도 절반.)
2. **체커보드 영상** (권장): `take.calibrate_checkerboard`가 10 프레임마다 9 × 6 내부 코너(칸 25 mm)를 찾고(최대 40장),
   `cv2.calibrateCamera`(Zhang의 방법)로 K를 최소제곱으로 구합니다. **RMS 재투영 오차 < 1 px**이면 좋습니다.
3. **아무것도 없음**: ②에서 천의 모서리 4개로 초점거리를 추정합니다 (직사각형의 두 소실점 방향이 직교한다는 조건). 우리 영상에서 25 % 틀렸습니다.

In [ ]:
if BOARD_VIDEO and BOARD_VIDEO.endswith('.json'):                    # 1. 이미 아는 K
    cj = json.load(open(BOARD_VIDEO)); k = W / cj['width']           # 원래 해상도 -> 960 폭
    K = np.array([[cj['fx'] * k, 0, cj['cx'] * k], [0, cj['fy'] * k, cj['cy'] * k], [0, 0, 1.]]); K_SOURCE = 'camera.json'
    print('camera.json:', K.round(1).tolist())
elif BOARD_VIDEO:                                                    # 2. 체커보드
    K, rms, n = take.calibrate_checkerboard(BOARD_VIDEO, width=W); K_SOURCE = 'checkerboard'
    print(f'checkerboard: {n} views, RMS {rms:.2f} px (good < 1)  fx {K[0, 0]:.1f} fy {K[1, 1]:.1f} cx {K[0, 2]:.1f} cy {K[1, 2]:.1f}')
else: K, K_SOURCE = None, 'cloth corners (after step 2)'             # 3. ②의 모서리로 (아래 2-2)
if K is not None:
    hfov = 2 * np.degrees(np.arctan(W / 2 / K[0, 0]))
    print(f'horizontal field of view {hfov:.0f} deg')

> **확인.** cx, cy가 영상 중심(480, 270) 근처인지, 수평 화각이 상식적인지 보세요. 화각이 너무 좁으면(예: 40° 미만) 줌이나 손떨림 보정의 잘라내기를 의심하세요.

### 1-3 프레임 고르기
아래 그림을 보고 네 가지 프레임 번호를 정해 **다음 셀에 적으세요** (기본값은 교수님 휴대폰 영상의 값). 그림의 번호는 영상 전체의 프레임 번호입니다.

| 변수 | 뜻 | 왜 |
|---|---|---|
| `REG` | **등록 프레임**: 천을 평평하게 펴서 카메라 쪽으로 든 프레임 | 네 모서리가 모두 보여야 모서리 → 격자를 만들 수 있음 |
| `START` | 두 손이 천을 잡고 있는 첫 프레임 | 손이 천을 잡기 전은 시뮬레이터가 재현할 수 없음 |
| `END` | 마지막 프레임 | 손을 놓거나 화면 밖으로 나가기 전 |
| `FIT_END` | 여기까지로 맞추고, 그 뒤는 **검증(held-out)** | 맞추지 않은 구간도 예측하는지 = 과적합 확인 |

In [ ]:
step = max(1, T // 16)
viz.show(viz.sheet([frames[i] for i in range(0, T, step)], cols=4, w=320, labels=[f'frame {i}' for i in range(0, T, step)]), 'pick the frames', (18, 8))

In [ ]:
# TODO: 위 그림을 보고 고르세요 (아래 값은 교수님 휴대폰 영상 my_take.mp4의 값)
REG   = 70      # 등록 프레임: 천을 카메라 쪽으로 평평하게 펴서 든 프레임 (네 모서리가 다 보임)
START = 48      # 두 손이 천을 잡고 있는 첫 프레임
END   = 328     # 마지막 프레임
FIT_END = 220   # 이 프레임까지로 맞추고, 그 뒤는 검증 (held-out)
q = REG; viz.show(frames[REG], f'registration frame {REG}')
print(f'fit {START}-{FIT_END}: {t[FIT_END] - t[START]:.1f} s,  held-out {FIT_END}-{END}: {t[END] - t[FIT_END]:.1f} s')

## 2. 인식 (M2) — 사전학습된 모델 네 개

②의 목표는 **"천 위의 정해진 점들이 매 프레임 어디에 있는가 (픽셀 + 깊이)"**입니다. 각 점이 천의 어디(`uv`)인지 알아야
시뮬레이터의 같은 점과 비교할 수 있습니다. 그래서 순서가 이렇습니다:

```
등록 프레임 ─ DINO ─▶ 상자 ─ SAM 2 ─▶ 마스크 ─ 사각형 ─▶ 모서리 c0..c3 ─ homography ─▶ 격자점 121개의 픽셀 (uv를 앎)
                                                                                        │
                       모든 프레임 ◀──────────────────── CoTracker3 (앞으로 + 뒤로) ─────┘
                       모든 프레임 ─ Depth Anything V2 ─▶ 깊이 지도
```

모든 모델은 `perceive.py`에서 처음 부를 때 한 번 받아 GPU에 올리고(`_load`가 캐시), 이후에는 재사용합니다.

### 2-1 Grounding DINO: 글로 물체 찾기
**모델.** 글과 이미지를 함께 넣는 open-set 검출기입니다. 글(BERT)과 이미지(Swin) 특징을 cross-attention으로 섞고, 글과 가장 관련 있는
위치에서 상자를 만들어 **상자마다 글의 단어와 맞는 점수**를 냅니다. 학습 때 없던 "white cloth" 같은 표현도 찾습니다.

**`perceive.detect_box(img, prompt)` 안에서** (`IDEA-Research/grounding-dino-tiny`, transformers):
1. processor가 이미지와 글을 텐서로 → 모델 실행
2. `post_process_grounded_object_detection(threshold=0.3, text_threshold=0.25)`: 점수 0.3 미만 상자는 버림
3. 남은 상자 중 **점수가 가장 높은 하나**를 돌려줌 (없으면 에러 → 프롬프트를 바꾸라는 뜻)

**프롬프트 규칙**: 소문자, 마침표로 끝, 여러 물체는 `"cloth . towel ."`처럼 ` . `로 구분.

In [ ]:
PROMPT = 'white cloth.'
box, score = perceive.detect_box(frames[REG], PROMPT)                 # box: (x0, y0, x1, y1) 픽셀, score: 0..1
im = frames[REG].copy(); cv2.rectangle(im, tuple(box[:2].astype(int)), tuple(box[2:].astype(int)), (255, 80, 0), 3)
viz.show(im, f'Grounding DINO: "{PROMPT}" score {score:.2f}')
print('box', box.round(1), 'score', round(score, 3))

> **확인.** 상자가 천 전체를 감싸는지 보세요 (교수님 영상: 점수 약 0.83). 사람이나 배경을 잡으면 `PROMPT`를 `'cloth.'`, `'towel.'` 등으로 바꿔 다시 실행.

### 2-2 SAM 2.1: 상자 → 마스크 → 모서리 4개
**모델.** SAM은 무거운 이미지 인코더를 한 번 돌리고, 가벼운 디코더가 프롬프트(점, 상자, 마스크)마다 마스크를 냅니다.
SAM 2는 여기에 메모리를 더해 영상으로 확장한 모델이고, 한 장만 넣으면 SAM처럼 동작합니다.

**`perceive.segment_frame(img, box, points)` 안에서** (`facebook/sam2.1-hiera-small`):
- 프롬프트 = DINO 상자 + **상자 중심을 양(+)의 점**으로 하나 더 (천 안쪽이라는 힌트)
- `multimask_output=False`: 후보 3개 대신 마스크 하나만
- 결과를 원래 해상도로 올려 `bool (H, W)`

**`take.auto_corners(mask)` 안에서**:
1. 마스크의 가장 큰 외곽선 → **볼록 껍질**(convex hull: 손가락·주름으로 오목한 곳을 메움)
2. `cv2.approxPolyDP`의 허용 오차를 0.5 %에서 10 %까지 늘리며 **꼭짓점이 정확히 4개**인 다각형을 찾음
3. y가 작은 두 개 = 위(손이 잡은 모서리), 그중 왼쪽이 **c0**, 오른쪽이 **c3**; 아래 두 개 중 왼쪽 **c1**, 오른쪽 **c2**

```
   c0 ●━━━━━━━━━━━━━━● c3      ← 두 손이 잡은 변 (HEIGHT_CM)
      ┃              ┃
      ┃     천       ┃          WIDTH_CM
      ┃              ┃
   c1 ●━━━━━━━━━━━━━━● c2
```

In [ ]:
ctr = np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2])        # 상자 중심 = 천 안쪽의 점 (양의 프롬프트)
mask_reg = perceive.segment_frame(frames[REG], box, [ctr])             # (H, W) bool
C = take.auto_corners(mask_reg)                                         # (4, 2): c0, c1, c2, c3 픽셀
# 틀렸으면 직접 넣으세요 (c0, c1, c2, c3 순서, 960 폭 영상의 픽셀):  C = np.array([[x0, y0], [x1, y1], [x2, y2], [x3, y3]], float)
viz.show(viz.draw_corners(viz.overlay_mask(frames[REG], mask_reg), C), 'SAM 2 mask and the corners c0..c3')
print('mask covers', round(mask_reg.mean() * 100, 1), '% of the frame | corners', C.round(0).tolist())

> **확인.** 마스크가 손·팔을 포함하지 않는지, c0가 **왼손**, c3가 **오른손** 위치인지 보세요.
> 손가락이 모서리를 가리면 자동 모서리가 손가락 쪽으로 밀립니다 (교수님 영상: 아래 모서리는 손으로 찍은 위치와 2 px, 위 모서리는 14–20 px 차이).
> 많이 틀리면 위 주석의 `C = ...`로 직접 넣으세요.

체커보드도 `camera.json`도 없을 때만, 여기서 천의 모서리로 초점거리를 추정합니다 (주점은 영상 중심으로 가정).

In [ ]:
if K is None:                       # 체커보드가 없을 때: 천의 소실점으로 초점거리 추정 (덜 정확: 실험에서 25 % 틀림)
    f_est = take.focal_from_corners(C, (W, H)); K = np.array([[f_est, 0, W / 2], [0, f_est, H / 2], [0, 0, 1.]]); print('focal from the cloth corners', round(f_est, 1))
reg = dict(frame=REG, corners_px=C.tolist(), held=[0, 3], start=START, end=END, fit_end=FIT_END)   # 등록 정보: ③에서 씀

**(선택) SAM 2 영상 분할.** `segment_video`는 등록 프레임의 상자 프롬프트를 **memory bank**에 넣고 앞으로, 뒤로 전파해 모든 프레임의 마스크를 만듭니다.
이후 단계에는 쓰지 않고 **확인용**입니다 (천이 접히거나 가려질 때 마스크가 어떻게 되는지). T4에서 수 분, CPU에서는 프레임당 약 16초.

In [ ]:
RUN_SAM_VIDEO = False        # True: 모든 프레임의 마스크 (확인용 영상, T4에서 수 분)
if RUN_SAM_VIDEO:
    masks = perceive.segment_video(frames[START:END + 1], REG - START, box, [ctr])     # (T', H, W) bool
    display(Video(viz.write_video(f'{OUT}/masks.mp4', [viz.overlay_mask(frames[START + i], m) for i, m in enumerate(masks)]), embed=True, width=640))

### 2-3 추적할 점: uv 격자 → homography → 픽셀
천의 texture 좌표 (u, v) ∈ [0, 1]²에 **11 × 11 = 121개**의 격자점을 놓습니다. 각 점이 천의 어디인지(uv) 알고 있으므로,
나중에 시뮬레이터 메쉬에서 **같은 uv의 점**을 찾아 비교할 수 있습니다 (`at_uv`: 그 uv를 포함하는 삼각형의 barycentric 보간).

**homography** `Hm` (3 × 3)는 평면 → 평면 사영 변환입니다. 모서리 4쌍(uv ↔ 픽셀)으로 정확히 정해집니다:

| 모서리 | uv | 픽셀 |
|---|---|---|
| c0 | (0, 0) | `C[0]` |
| c1 | (1, 0) | `C[1]` |
| c2 | (1, 1) | `C[2]` |
| c3 | (0, 1) | `C[3]` |

등록 프레임에서 천이 **평평**해야 하는 이유가 이것입니다: 평면일 때만 homography가 격자를 제자리에 놓습니다.

In [ ]:
GRID = 11
g = np.linspace(0, 1, GRID); uv = np.array([(u, v) for v in g for u in g])              # (121, 2)
corner_uv = np.float32([[0, 0], [1, 0], [1, 1], [0, 1]])                                 # c0, c1, c2, c3 의 uv
Hm = cv2.getPerspectiveTransform(corner_uv, C.astype(np.float32))                        # uv -> 등록 프레임 픽셀
px = cv2.perspectiveTransform(uv[None].astype(np.float32), Hm)[0]                        # (121, 2) 격자점의 픽셀
corner_ids = np.array([int(np.argmin(np.linalg.norm(uv - c, axis=1))) for c in corner_uv])   # 모서리에 해당하는 격자점 번호
held_ids = corner_ids[[0, 3]]                                                            # 손이 잡은 c0, c3
viz.show(viz.draw_points(frames[REG], px, np.ones(len(px), bool), held_ids), f'{len(px)} grid points on the registration frame (blue = held corners)')

### 2-4 CoTracker3: 121개 점을 모든 프레임에서 추적
**모델.** 점 여러 개를 **함께** 보는 transformer 추적기입니다. 점들끼리 서로 참고하므로 무늬가 적은 곳이나 잠깐 가려진 점도
주변 점의 움직임으로 위치를 추정하고, 각 점이 **보이는지(visibility)**도 함께 냅니다.

**`perceive.track_points(frames, q, px)` 안에서:**
1. 영상을 **절반 크기**로 줄임 (`scale=0.5`: 메모리·속도. 결과 좌표는 다시 2배)
2. **online 모델**(`cotracker3_online`, torch.hub)은 짧은 창(window) 단위로 영상을 읽어 GPU 메모리가 일정합니다.
   offline 모델은 영상 전체를 한꺼번에 올려 T4(16 GB)에서 약 400 프레임에 메모리가 부족했습니다.
3. online 모델은 **앞으로만** 갑니다. 그래서 등록 프레임 q에서 **앞으로 한 번**(`frames[q:]`), **뒤집은 영상으로 한 번**(`frames[:q+1][::-1]`) 돌려 이어 붙입니다.
4. GPU에서는 fp16(autocast)로 계산, `vis > 0.5`이면 보임.

**출력**: `xy (T', 121, 2)` 픽셀, `vis (T', 121)` bool.

In [ ]:
seg = frames[START:END + 1]; t0 = time.time()                                            # 손이 천을 잡은 구간만
xy, vis = perceive.track_points(seg, REG - START, px)                                    # q = 구간 안에서 등록 프레임의 위치
tracks = dict(frames=np.arange(START, END + 1), xy=xy, vis=vis, uv=uv, corner_ids=corner_ids)
np.savez(f'{OUT}/tracks.npz', **tracks)
print(f'tracked {len(uv)} points, visible {vis.mean() * 100:.0f} %, {time.time() - t0:.0f} s  | xy {xy.shape}, vis {vis.shape}')

In [ ]:
pick = np.linspace(0, len(seg) - 1, 8).astype(int)
viz.show(viz.sheet([viz.draw_points(seg[i], xy[i], vis[i], held_ids) for i in pick], labels=[f'frame {START + i}' for i in pick]), 'green seen, red hidden, blue held corners', (18, 5.5))
display(Video(viz.write_video(f'{OUT}/tracks.mp4', [viz.draw_points(seg[i], xy[i], vis[i], held_ids) for i in range(len(seg))]), embed=True, width=640))

> **확인.** 영상에서 점들이 천에 **붙어서** 움직이는지 보세요. 점이 배경이나 팔로 미끄러지면 그 구간은 ⑤의 오차를 키웁니다.
> 손이 잡은 모서리(파랑)는 손가락에 가려 빨강(hidden)이 되기 쉽지만, ③에서 **손의 위치로는 계속 씁니다** (가려져도 추적기가 위치를 추정).
> 보이는 비율이 낮으면(예: 60 % 미만) 천이 많이 접혔거나 화면 밖으로 나간 구간이 있는지 보세요.

### 2-5 Depth Anything V2 Metric: 사진 한 장 → 미터 단위 깊이
**모델.** 한 장의 사진에서 픽셀마다 거리(m)를 예측합니다 (실내 데이터로 학습한 Metric-Indoor 판, Base 크기).

**`perceive.metric_depth(frames, model)` 안에서:** 프레임마다 모델 실행 → 깊이 지도를 **영상의 절반 해상도**(270 × 480)로 보간 → float16으로 저장.
출력 `depth (T', H/2, W/2)`와 그 프레임 번호 `didx`.

**알려진 한계** (같은 영상을 RealSense depth 카메라와 비교한 결과):

| | 천(약 0.65 m)에서 예측 / 실제 | 프레임마다 scale 하나를 맞춘 뒤 |
|---|---|---|
| Small | 1.2–2.6배, 프레임마다 다름 | 약 6.3 % (35 mm) |
| **Base (여기서 씀)** | 1.3–1.7배, 프레임마다 다름 | 약 3.5 % (21 mm) |

배경(약 3 m)은 거의 맞는데 가까운 천은 멀게 봅니다. **scale은 틀리지만 모양(상대 깊이)은 꽤 맞으므로**, ③에서 천의 실제 크기로 프레임마다 scale을 맞춥니다.

In [ ]:
DEPTH_MODEL = 'depth-anything/Depth-Anything-V2-Metric-Indoor-Base-hf'
t0 = time.time(); depth, didx = perceive.metric_depth(seg, DEPTH_MODEL); print(f'{len(depth)} depth maps, {time.time() - t0:.0f} s  | depth {depth.shape} {depth.dtype}')
np.savez_compressed(f'{OUT}/depth.npz', depth=depth, frames=didx + START)
viz.show(viz.sheet([cv2.resize(seg[i], (W // 2, H // 2)) for i in pick[:4]] + [viz.depth_color(depth[i]) for i in pick[:4]]), 'metric depth (blue 0.3 m -> red 3 m)', (18, 5.5))

In [ ]:
# 등록 프레임에서 천의 깊이를 숫자로: 카메라에서 천까지 실제 거리(줄자로 잰 값)와 비교해 보세요
d_reg = depth[REG - START].astype(float); m_half = cv2.resize(mask_reg.astype(np.uint8), (W // 2, H // 2)) > 0
print(f'registration frame: median depth on the cloth {np.median(d_reg[m_half]):.2f} m, background {np.median(d_reg[~m_half]):.2f} m')

## 3. 3-D로 올리기 (M3, TODO 3개)

**무엇을.** 2-D 추적 + 깊이 → 중력 방향이 정해진 **월드 좌표계(z가 위, 바닥이 z = 0)의 3-D 점**. 시뮬레이터는 중력이 −z이고 단위가 m이므로
관찰을 같은 좌표계·같은 단위로 맞춰야 합니다.

`lift.lift_take`가 하는 일 (아래 굵은 것이 TODO 함수):

| 순서 | 처리 | 이유 |
|---|---|---|
| 1 | 각 점의 깊이 = 주변 5 × 5 창의 **25번째 백분위수** (`sample`) | 천 가장자리의 점이 뒤 배경(수 m)의 깊이를 집지 않게. 최솟값이 아닌 백분위수: 잡음에 덜 흔들림 |
| 2 | **`backproject`**: 픽셀 + 깊이 → 카메라 좌표 3-D | 핀홀 모델 (아래 식) |
| 3 | 이웃 격자점 쌍과 그 **실제 거리** (`neighbour_pairs`) | uv 차이 × 천 크기 (u 방향 c0→c1 이웃 = WIDTH_CM / 10, v 방향 c0→c3 이웃 = HEIGHT_CM / 10: ⓪에서 만든 메쉬의 크기) |
| 4 | **`per_frame_scale`**: 프레임마다 (실제 거리 / 3-D 거리)의 **중앙값** | 깊이 모델의 scale이 프레임마다 틀리므로. 중앙값: 접힌 곳·잘못 추적된 점에 안 흔들림 |
| 5 | scale을 5 프레임 중앙값 필터로 부드럽게 (`smooth`) | 한 프레임짜리 튀는 값 제거 |
| 6 | 바닥 픽셀(천이 한 번도 안 들어간 아래쪽 행)의 깊이를 **프레임 중앙값**으로 → 3-D | 카메라가 고정이라 바닥 깊이는 매 프레임 같음 → 잡음이 평균에서 사라짐. 바닥의 **높이**(z = 0)는 항상 여기서 |
| 7 | **중력 방향**: `GRAVITY = 'lines'`(권장)면 영상 속 **세로선**으로 (3-3), `'floor'`면 **`floor_frame`**: RANSAC 평면(허용 2 cm)의 법선 | 시뮬레이터의 중력 방향과 맞춤 |
| 8 | 점과 손 궤적을 시간으로 부드럽게 (점 5 프레임, 손 `HAND_SMOOTH` = 15 프레임 이동평균) | 깊이 잡음으로 손이 카메라 방향으로 1–2 cm 떨면 시뮬레이션 천도 흔들림 |

**카메라 좌표계**: x 오른쪽, **y 아래**, z 앞(카메라가 보는 방향). 그래서 "위"는 카메라 y가 음수인 쪽입니다.

$$\mathbf{X}_{cam} = \Big(\tfrac{u-c_x}{f_x}z,\ \tfrac{v-c_y}{f_y}z,\ z\Big),\qquad s_t = \operatorname{median}_{(i,j)\ \text{보임}} \frac{L_{ij}}{\lVert \mathbf{X}_i - \mathbf{X}_j \rVert},\qquad \mathbf{X}_{world} = R\,(s_t\mathbf{X}_{cam}) + \mathbf{t}_w$$

### 3-1 TODO 함수
- `backproject(xy (...,2), z (...), K) -> (...,3)`: 위 식 그대로. `...`은 아무 모양이나 되도록 `xy[..., 0]`처럼 씁니다.
- `per_frame_scale(X (T,P,3), vis (T,P), pairs (Q,2), L (Q,)) -> s (T,)`: 두 점이 **모두 보이는** 쌍만, 그런 쌍이 8개 미만이면 NaN.
- `floor_frame(P_floor (N,3)) -> R, tw, 바닥 점 비율`:
  1. `lift.fit_plane_ransac`로 법선 n, 평면 위 점 c
  2. 카메라 y가 아래이므로 `n[1] > 0`이면 n을 뒤집어 n이 **위**를 향하게
  3. 월드 x = 카메라 x축을 바닥에 투영해 정규화, 월드 y = n × x (오른손 좌표계), 월드 z = n
  4. `R`의 행이 (x, y, n) → `world = R @ cam + tw`, 바닥 위의 c가 z = 0이 되도록 `tw = -R @ c`

In [ ]:
# ===== TODO (M3): 세 함수를 채우세요 =====
def backproject(xy, z, K):
    """xy (..., 2) 픽셀, z (...) 광축 방향 깊이 (m), K 3x3 -> (..., 3) 카메라 좌표 점 (x 오른쪽, y 아래, z 앞)."""
    raise NotImplementedError('backproject')

def per_frame_scale(X, vis, pairs, L, min_pairs=8):
    """X (T, P, 3) 원래 depth로 올린 점, vis (T, P), pairs (Q, 2) 이웃 점 쌍, L (Q,) 그 쌍의 실제 거리 (m).
    -> s (T,): 프레임마다, 두 점이 모두 보이는 쌍들의 (실제 거리 / 잰 3-D 거리)의 중앙값. 보이는 쌍이 min_pairs보다 적으면 NaN."""
    raise NotImplementedError('per_frame_scale')

def floor_frame(P_floor):
    """카메라 좌표의 바닥 점들 -> (R, t, inlier 비율), world = R @ cam + t: z가 위, 바닥이 z = 0.
    lift.fit_plane_ransac(P_floor) -> (n, c, frac)를 쓰세요. 카메라 y는 아래쪽이라 '위'는 카메라 y가 음수인 방향.
    R의 행: x = 카메라 x를 바닥에 투영해 정규화, z = n (위), y = z × x."""
    raise NotImplementedError('floor_frame')

lift.backproject, lift.per_frame_scale, lift.floor_frame = backproject, per_frame_scale, floor_frame     # 내 함수로 바꿔 끼우기

### 3-2 테스트
`check.run()`은 정답을 아는 작은 예제로 TODO 함수 7개를 시험합니다 (③ 3개 + ⑤ 4개, 모두 13개 시험). 지금은 ⑤를 아직 안 바꿨으므로
**⑤ 쪽 시험은 저장소에 있는 함수로** 돕니다 (학생 저장소에서는 비어 있어 `not implemented` FAIL이 정상, 교수자 저장소에서는 PASS). ③ 시험이 PASS인지 보세요.

In [ ]:
ok, _ = check.run()

### 3-3 중력 방향: 영상 속 세로선 (권장) 또는 바닥 평면

시뮬레이터의 중력은 −z이므로 관찰의 "위"가 틀리면 **어떤 물성으로도** 영상을 따라갈 수 없습니다
(교수님 영상 T01: 중력 방향이 13° 틀렸을 때 손실 54 mm → 고친 뒤 42 mm. young을 1000배 바꾸는 것보다 큰 차이).

**바닥 평면 (`floor_frame`)의 약점.** 바닥 픽셀 = 천이 한 번도 들어가지 않은 **아래쪽 행**입니다. 천이 화면 아래 끝까지 내려오면 얇은 띠(20행, 시야각 약 2.5°)만 남고,
그 띠의 3-D 점들은 면이 아니라 **카메라에서 뻗은 선**처럼 놓입니다. 깊이 오차는 점을 시선 방향으로만 밀기 때문에 평면이 그 선을 품고 기울어집니다
(T01: 기울기 17°, 카메라 높이 −1 cm: 불가능한 값).

**세로선.** 방 안의 수직선(문틀, 가구 모서리, 벽 모서리)은 모두 중력과 평행합니다. 깊이는 전혀 쓰지 않고 **K만** 씁니다.
- 영상의 선분 (p₁, p₂) → 직선 l = p̃₁ × p̃₂ (동차 좌표). 이 직선과 카메라 중심을 지나는 평면의 법선은 **m = Kᵀ l**
- 3-D 수직 방향 d는 그런 평면들 안에 **모두** 있어야 함: m·d = 0 → 선 2개 이상의 m을 쌓은 행렬의 **가장 작은 특이벡터** (SVD) = d
- 선마다의 잔차(도) = d가 그 평면에서 벗어난 각도. **2°를 넘는 선은 빼고 다시** (사선, 천, 팔을 잘못 고른 것)

`lift.vertical_candidates`가 거의 수직인 긴 선분을 찾아 번호를 붙입니다. 그림을 보고 **진짜 수직인 것만** `USE`에 적으세요.

> **⚠ `USE`를 기본값(후보 앞 8개)으로 두면 안 됩니다.** 후보에는 천 가장자리, 팔, 사선도 섞입니다. 선마다의 잔차가 모두 2° 아래여도 틀릴 수 있습니다.
> 교수님 영상 T01에서 기본값은 **기울기 8.2°**를 주고, 그러면 ⑤b의 강성 곡선이 끝까지 오르지 않아 **상한을 못 정합니다** (`inf`, 손실 바닥 24.0 mm, tol 4.6 mm).
> 문틀 · 가구 모서리만 고르면 **4.6°** (up = `[-0.022, -0.997, -0.078]`), 이때 공기 저항 0.94, 상한 4.6e4 Pa (바닥 22.5 mm, tol 2.0 mm).
> 기울기가 삼각대를 둔 모양과 맞는지, 고른 선이 그림에서 정말 문틀 · 모서리인지 꼭 눈으로 확인하세요.

In [ ]:
GRAVITY = 'lines'      # TODO: 'lines' (영상 속 세로선, 권장) | 'floor' (바닥 평면, 세로선이 없을 때)
HAND_SMOOTH = 15       # 손 궤적 이동평균 창 (프레임, 30 fps에서 0.5 s)
LINE_FRAME = END       # 세로선이 천 · 팔에 가리지 않는 프레임
cand = lift.vertical_candidates(frames[LINE_FRAME])                     # (N, 4): x1 y1 x2 y2, 긴 것부터
im = frames[LINE_FRAME].copy()
for i, (x1, y1, x2, y2) in enumerate(cand.astype(int)):
    cv2.line(im, (x1, y1), (x2, y2), (255, 0, 0), 2); cv2.putText(im, str(i), (x1 + 4, (y1 + y2) // 2), 0, .7, (255, 255, 0), 2)
viz.show(im, f'vertical line candidates (frame {LINE_FRAME})', (14, 8))

In [ ]:
USE = list(range(min(8, len(cand))))   # TODO: 위 그림에서 '진짜 수직'인 선 번호만 (문틀, 가구 모서리. 천 · 팔 · 사선은 빼기). 기본값 그대로 두지 말 것 (T01: 8.2° → 상한 inf)
EXTRA = []                             # 후보에 없으면 직접 [[x1, y1, x2, y2], ...] (960 폭 픽셀)
UP_CHECKED = None                      # 이미 확인한 '위' 벡터가 있으면 직접 (교수님 영상 T01: [-0.022, -0.997, -0.078], 4.6°)
if GRAVITY == 'lines':
    segs = np.array([cand[i] for i in USE] + EXTRA, float)
    up, res_deg = lift.up_from_lines(segs, K)                          # up: 카메라 좌표의 '위' (단위 벡터)
    print(f'up (camera coords) {up.round(3)}  -> camera tilted {np.degrees(np.arccos(-up[1])):.1f} deg from level')
    print('residual per line (deg):', res_deg.round(2), ' <- 2도 넘는 선은 USE에서 빼고 다시')
    if UP_CHECKED is not None:
        up = np.array(UP_CHECKED, float); up /= np.linalg.norm(up)
        print(f'using UP_CHECKED {up.round(3)}  -> tilt {np.degrees(np.arccos(-up[1])):.1f} deg')
else: up = None

### 3-4 관찰을 3-D로
`lift_take`의 출력 `obs`가 ④, ⑤의 입력 전부입니다.

| 키 | 모양 | 뜻 |
|---|---|---|
| `t` | (T',) | 시각 (s) |
| `P` | (T', 121, 3) | 천의 점, 월드 좌표 (m). 안 보이면 NaN |
| `xy` | (T', 121, 2) | 추적된 픽셀 (안 보이면 NaN): ⑤의 **영상 위 비교**에 씀 |
| `hands` | (T', 2, 3) | 손이 잡은 c0, c3의 궤적 (가려져도 있음, `HAND_SMOOTH` 프레임으로 부드럽게 함): 시뮬레이터의 손 |
| `grasp` | (2,) | 손이 잡은 메쉬 꼭짓점 번호 |
| `fit` | (T',) bool | 맞추기 구간 (FIT_END까지) |
| `R`, `tw`, `K` | | 카메라 ↔ 월드 (⑤에서 시뮬레이션을 영상에 투영) |
| `scale` | (T',) | 프레임마다 깊이 scale |

In [ ]:
obs, diag = lift.lift_take(tracks, depth, K, reg, mesh, t, depth_frames=didx + START, up=up, hand_smooth=HAND_SMOOTH)
print(f'gravity from the {diag["gravity"]}: tilt {diag["tilt_deg"]:.1f} deg | camera {diag["camera_height"]:.2f} m above the floor '
      f'(floor rows {diag["floor_rows"]}) | points seen {diag["seen"] * 100:.0f} % | hand jitter removed {diag["hand_jitter_mm"]:.1f} mm rms')
print('obs:', {k: getattr(v, 'shape', v) for k, v in obs.items() if k in ('t', 'P', 'xy', 'hands', 'fit')})
print(f'depth scale per frame: median {np.nanmedian(diag["scale"]):.2f}, range {np.nanmin(diag["scale"]):.2f}-{np.nanmax(diag["scale"]):.2f}')
viz.plot_lift(obs, diag, 'lifted observation')

> **확인 (그림과 숫자).**
> - **scale**: 1보다 작으면 깊이 모델이 천을 멀게 본 것 (Base: 약 0.6–0.8 기대). 곡선이 부드럽게 변하는지.
> - **기울기**: 삼각대를 수평으로 두었다면 몇 도 안이어야 함. 세로선과 바닥 평면이 10° 넘게 다르면 바닥 평면 쪽을 의심.
> - **카메라 높이**: 실제로 카메라를 둔 높이(삼각대, 책상)와 비슷한지. 0 근처나 음수이면 바닥 띠가 너무 얇은 것 (`floor_rows`가 20 근처).
> - **손 사이 거리**: 두 손이 `HEIGHT_CM` 변의 양 끝을 잡으므로, 천을 펴서 들 때 약 `HEIGHT_CM`, 손을 좁히면 더 작아야 함. `HEIGHT_CM`를 크게 넘으면 scale이나 추적이 틀린 것 (천은 안 늘어남).
> - **3-D 점**: 바닥(z = 0)보다 위에, 손이 가장 높은 곳에 있는지.

## 4. 시뮬레이터 (MuJoCo 4.0.1)

**천 모델** (`sim/cloth_sim.py`, `ClothSim(mesh_dir, young, air=...)`):
- 메쉬 = MuJoCo **flex** (⓪에서 만든 메쉬, 질량 `CLOTH_MASS_G`)
- **edge equality**: 모든 변의 길이를 유지 → 직물처럼 늘어나지 않음
- **elasticity `elastic2d="bend"`, `young`**: 굽힘 강성 D = young × t³ / (12(1 − ν²)), 두께 t = 2 mm 고정 → **⑤b에서 상한을 잼**
- **공기 저항 `air`** (N·s/m³, 천 1 m²당): 꼭짓점마다 F = −air × (천 면적 / 꼭짓점 수) × v → **⑤a에서 찾음**
  - 가벼운 천은 무게에 비해 면적이 커서 공기 저항이 중력만큼 중요합니다 (15 g, 0.12 m²).
  - flex의 `damping`은 **변형** 속도만 줄이고 천 **전체가 흔들리는** 운동은 줄이지 않습니다. 공기가 없으면 시뮬레이션 천이 계속 찰랑거립니다
    (T01: 영상보다 4배 큰 떨림. air = 0.85를 넣자 영상 수준이 되고 손실이 절반).
  - 운동이 줄어드는 시간 τ = 면적당 질량 / air ≈ 0.13 / 0.85 ≈ 0.15 s. 평평한 천이 떨어지는 속도로 환산하면 g·τ ≈ 1.5 m/s (실제 얇은 천과 비슷).
- 손 = mocap 몸체 2개. 잡은 모서리에서 가장 가까운 꼭짓점 3개를 손에 **connect** 제약으로 연결
- Euler 적분, 1 ms 간격. 시작 전 1초 동안 첫 손 위치에서 매달려 자리를 잡음(settle). 처음 모양을 평면에서 1 cm 휘게(bow) 둠

**`fit.simulate(obs, MESH, young, air)`**: `obs['hands']`를 시각 `obs['t']`에 맞춰 손을 움직이며 rollout → 모든 꼭짓점 `X (T', N, 3)`,
그리고 `at_uv`로 추적점과 **같은 uv**의 시뮬레이션 점 `S (T', 121, 3)`.

아래는 같은 손 움직임에서 **공기 저항이 없을 때 / 있을 때**, **부드러운 천 / 아주 뻣뻣한 천**입니다. 어느 차이가 더 크게 보이나요?

In [ ]:
views, labels = [], []
for y, a in ((1e3, 0.), (1e3, 0.85), (3e5, 0.85)):
    S, X = fit.simulate(obs, MESH, y, a)                                # S (T', 121, 3) 추적점과 같은 uv, X (T', N, 3) 메쉬 전체
    ov = viz.overlay_sim(frames, obs, X, mesh['F'], label=f'young {y:.0e}, air {a}')      # 시뮬레이션 메쉬를 K, R, tw로 영상에 투영
    views += [ov[i] for i in np.linspace(0, len(ov) - 1, 4).astype(int)]
viz.show(viz.sheet(views, cols=4), 'rows: (young 1e3, no air) / (1e3, air 0.85) / (3e5, air 0.85); orange = simulation, green = tracked', (18, 10))

## 5. 공기 저항 → 강성 상한 → 강성 값 (M4, TODO 4개)

**손실.** 추적한 픽셀과, 시뮬레이션 천을 카메라로 투영한 픽셀 사이의 평균 거리 (천까지 거리에서의 mm로 환산).
맞추기 구간(`FIT_END`까지)으로 정하고, 그 뒤(held-out)는 확인에만 씁니다.

**왜 순서가 공기 → 강성인가.** 두 물성이 영상에 남기는 흔적이 다릅니다 (T01, 로컬에서 잰 값):

| | 바꾼 범위 | 손실 변화 | 모양 |
|---|---|---|---|
| 공기 저항 | 0 → 0.6 → 12 | 33 → **22** → 40 mm | **U자**, 최솟값이 분명 |
| 굽힘 강성 | 1e2 → 3e4 | 22.6 – 24 mm (잡음 수준) | **평평** |
| 굽힘 강성 | 3e4 → 3e5 | 24 → 27 mm | 오르기 시작 |

그리고 공기 저항의 최솟값은 young을 1e3, 1e4, 1e5 어느 값에 고정해도 0.6–1.5로 같았습니다 (서로 섞이지 않음).
그래서 **young을 평평한 구간의 값(`YOUNG_REF` = 1e3.5)에 고정하고 공기 저항을 먼저** 찾고, 그 공기 저항에서 young을 훑습니다.

| 함수 | 입력 → 출력 | 할 일 |
|---|---|---|
| **`frame_loss(obs, sim)`** | (P, d), (P, d) → 숫자 | 관찰이 NaN이 아닌(보이는) 점만, 같은 uv끼리의 **평균 거리** (d = 2 픽셀 또는 3 m) |
| **`episode_loss(obs, sim, t, sel)`** | (T, P, d) × 2, (T,), (T,) bool → 숫자 | 고른 프레임의 frame_loss 평균. **처음 0.5초(`fit.SKIP_S`)는 뺌** |
| **`curve_min(x, y)`** | 격자 x (n,), 손실 y (n,) → x 하나 | ⑤a: 가장 낮은 점과 양쪽 두 점씩(최대 5점)에 맞춘 **최소제곱 포물선의 꼭짓점** (가장 낮은 점의 양옆 사이로 자름). 최솟점이 격자 끝이면 그 끝. 3점만 쓰면 넓고 평평한 바닥의 잡음에 끌려감 |
| **`upper_bound(x, y, tol)`** | 격자 x, 손실 y, 허용치 → x 하나 | ⑤b: **그 점부터 끝까지 모두 바닥 + tol 위**인 첫 격자점 x (평평한 구간의 잡음 한 점은 무시). 바닥 = **가장 낮은 3점의 평균** (`fit.FLOOR_K`; 운 좋게 낮은 한 점이 바닥을 정하면 평평한 구간 전체가 띠 밖으로 밀림). 마지막 점이 띠 안이면 `inf` (상한 없음). 두 격자점 사이를 보간하지 않는 이유: 손실이 오른 것을 **본** 곳만 배제할 수 있음 (합성 영상에서 보간한 상한은 정답보다 작았음) |

**왜 log 공간인가.** 두 물성 모두 배수로 변할 때 운동이 달라집니다. 그래서 x = log10(air), log10(young)으로 다룹니다.

In [ ]:
# ===== TODO (M4): 네 함수를 채우세요 =====
def frame_loss(obs, sim):
    """obs (P, d) 관찰된 점 (NaN 행 = 안 보임), sim (P, d) 같은 uv의 시뮬레이션 점 -> 보이는 점들의 평균 거리 (없으면 NaN).
    d = 3 (3-D, m) 또는 2 (영상 위, 픽셀) 모두 되게."""
    raise NotImplementedError('frame_loss')

def episode_loss(obs, sim, t, sel):
    """obs, sim (T, P, d), t (T,) s, sel (T,) bool -> 선택된 프레임들의 frame_loss 평균 (NaN 무시).
    처음 fit.SKIP_S 초는 빼세요 (시뮬레이터는 자기 늘어진 모양에서 시작하므로)."""
    raise NotImplementedError('episode_loss')

def curve_min(x, y):
    """x (n,) 증가하는 격자 (log10 air), y (n,) U자 손실 -> 최솟값의 x.
    가장 낮은 점 i와 양쪽 두 점씩(i-2..i+2, 격자 안만) 최대 5점에 최소제곱 포물선(np.polyfit 2차) -> 꼭짓점 -b/(2a), x[i-1]..x[i+1] 사이로 자르기.
    i가 격자 끝(0 또는 n-1)이면 x[i]. 포물선이 아래로 볼록하지 않으면(a <= 0) x[i]."""
    raise NotImplementedError('curve_min')

def upper_bound(x, y, tol):
    """x (n,) 증가하는 격자 (log10 young), y (n,) 평평하다가 오르는 손실, tol 허용치 (mm) -> 상한 x.
    lim = (가장 낮은 fit.FLOOR_K = 3개 y의 평균) + tol. 마지막 점이 lim 이하이면 float('inf') (상한 없음).
    아니면 '그 점부터 끝까지 모두 lim 위'인 첫 점 j의 x[j] (평평한 구간의 잡음 한 점은 무시됨). 보간하지 않음."""
    raise NotImplementedError('upper_bound')

fit.frame_loss, fit.episode_loss, fit.curve_min, fit.upper_bound = frame_loss, episode_loss, curve_min, upper_bound
ok, _ = check.run()                                                     # 13개 모두 PASS여야 다음으로

### 5a 공기 저항: young을 고정하고 U자 곡선의 최솟값

`fit.run_air(obs, MESH, young=fit.YOUNG_REF, airs=fit.AIR_GRID)`:
1. `AIR_GRID` = 0.1, 0.2, 0.4, 0.7, 1.2, 2, 3.5, 6 (약 0.25 decade 간격) 각각으로 시뮬레이션 → 맞추기 구간 손실 (영상 위, mm)
2. 시뮬레이션 8번을 **병렬**로 (`fit.WORKERS` 프로세스, 각 시뮬레이션은 CPU 한 개. Colab은 CPU 2개)
3. x = log10(air)에서 여러분의 `curve_min` → `A['air']`

**무엇을 기대하나.** 공기가 너무 적으면 시뮬레이션 천이 영상보다 오래 흔들리고, 너무 많으면 꿀 속처럼 늦게 따라갑니다. 그 사이에 **골짜기**가 있어야 합니다.

In [ ]:
t0 = time.time()
A = fit.run_air(obs, MESH, young=fit.YOUNG_REF, airs=fit.AIR_GRID)      # Colab T4(CPU 2개)에서 약 4–6분
print(f'-> air {A["air"]:.2f} N s/m^3 per m^2' + ('   (격자 끝! AIR_GRID를 넓혀 다시)' if A['at_edge'] else '') + f'   ({time.time() - t0:.0f} s)')
viz.plot_air(A)

> **확인 (U자 곡선).**
> - 최솟값이 **격자 안쪽**인가요? 끝(0.1 또는 6)이면 `fit.run_air(..., airs=...)`로 격자를 넓히세요. 6을 넘으면 손 궤적이 떨리는 것(손 떨림을 공기가 대신 눌러 주는 것)을 의심하세요.
> - 골짜기 깊이(가장자리와 최솟값의 차이)가 몇 mm인가요? **시뮬레이터 잡음(⑤b에서 잼, 1–3 mm)보다 충분히 커야** 값을 믿을 수 있습니다.
> - held-out 곡선도 같은 곳에서 낮은가요? 그렇다면 맞추기 구간에만 맞춘 것이 아닙니다.

### 5b 강성 상한: 공기 저항을 고정하고 young을 훑기

`fit.run_young(obs, MESH, air=A['air'], youngs=fit.YOUNG_GRID)`:
1. `YOUNG_GRID` = 1e2, 1e2.5, …, 1e5.5 (0.5 decade 간격, 8개)로 시뮬레이션 → 손실
2. **허용치 tol 정하기** (`fit.noise_mm`): 같은 young과 young × (1 + 10⁻⁹)로 두 번 시뮬레이션합니다. 측정으로는 절대 구분할 수 없는 차이인데도,
   천이 접히는 방향이 아주 작은 차이로 바뀌어(혼돈) 손실이 몇 mm 다릅니다. young 1e3, 1e4에서 잰 그 차이의 RMS = **시뮬레이터 잡음**,
   `tol = max(2 mm, 3 × 잡음)`
3. x = log10(young)에서 여러분의 `upper_bound(x, 손실, tol)` → 첫 상한
4. **좁히기**: 상한과 그 아래 격자점 사이에 young 2개를 더 시뮬레이션(병렬 한 번)하고 다시 `upper_bound` → `Y['young_upper']` (0.5 → 약 0.17 decade 간격)

**왜 최솟값이 아니라 상한인가.** 평평한 구간의 손실 차이는 잡음 크기라서, 그 안의 최솟값은 **우연히** 정해집니다
(v6까지의 방식: 교수님 영상에서 같은 데이터로 3.2e3, 1.5e3, 6.8e3이 나옴). 반면 "이보다 뻣뻣하면 영상과 확실히 다르다"는 잡음보다 큰 신호입니다.

In [ ]:
t0 = time.time()
Y = fit.run_young(obs, MESH, air=A['air'], youngs=fit.YOUNG_GRID)       # 시뮬레이션 8 + 잡음 4 + 좁히기 2번, 약 7–9분
print(f'-> loss floor {Y["floor_mm"]:.1f} mm, simulator noise {Y["noise_mm"]:.1f} mm -> tol {Y["tol_mm"]:.1f} mm')
print(f'-> young <= {Y["young_upper"]:.3g} Pa' + ('   (격자 안에서 오르지 않음: 상한을 못 정함)' if Y['no_bound'] else '') + f'   ({time.time() - t0:.0f} s)')
viz.plot_young(Y)

> **확인 (평평 → 오름).**
> - 왼쪽이 초록 띠(바닥 = 가장 낮은 3점의 평균, + tol) 안에서 **평평**하고, 오른쪽에서 띠를 **벗어나 계속 오르나요?** 그 경계가 상한입니다.
> - **상한 없음** (끝까지 띠 안): 이 영상의 동작으로는 3e5까지도 구분이 안 된 것. 손을 30 ↔ 15 cm로 천천히 모았다 펴는 구간이 있는 take로 다시 해 보세요.
> - **처음부터 오름** (상한이 1e2 근처): 너무 부드러운 쪽이 더 맞는다는 뜻. 질량(`CLOTH_MASS_G`)이나 중력 방향을 의심하세요.
> - held-out 곡선도 같은 곳에서 오르나요?

### 5c 강성 값 정하기: 상한 아래에서 log 중간

데이터가 말해 주는 것이 "정답 young은 [`fit.YOUNG_MIN` = 1e2, 상한] 안"뿐이라면, 그중 **최악의 경우 오차(decade)가 가장 작은** 값은 그 구간의 log 중간입니다
(어느 끝이 정답이어도 구간 폭의 절반만 틀림). `fit.choose_young(상한)`이 그 값이고, `Y['young']`에 들어 있습니다.

**이 값을 어디에 쓰나.** 다음 과제(로봇 천 접기)의 시뮬레이터에 넣을 강성입니다. 접기처럼 천을 크게 다루는 과제는 이 구간 안에서 강성을 바꿔도 결과가 거의 같았으므로,
**학습할 때는 [1e2, 상한]에서 무작위로 바꾸고(domain randomization), 기본값은 `Y['young']`**을 쓰면 됩니다. 공기 저항은 값 하나(`A['air']`)를 씁니다.

In [ ]:
YOUNG = Y['young']; AIR = A['air']                                     # TODO(보고서): 다른 값을 고른다면 여기서 바꾸고 이유를 적으세요
UPPER = min(Y['young_upper'], fit.YOUNG_GRID[-1])                     # 상한이 없으면(inf) 격자 끝으로
YOUNG_DR = (fit.YOUNG_MIN, UPPER)                                       # 다음 과제의 domain randomization 범위
print(f'air {AIR:.2f} | young upper bound {Y["young_upper"]:.3g} -> chosen young {YOUNG:.3g} Pa '
      f'(worst-case error {np.log10(UPPER / fit.YOUNG_MIN) / 2:.2f} decades) | randomize young in [{YOUNG_DR[0]:.0e}, {YOUNG_DR[1]:.2g}]')
cs = cloth_sim.ClothSim(MESH, young=YOUNG); rho_A = cs.mass / (mesh['width'] * mesh['height'])
D = YOUNG * cloth_sim.THICK ** 3 / (12 * (1 - .3 ** 2)); c_bend = (D / (rho_A * 9.81)) ** (1 / 3)
print(f'bending length of the chosen young {c_bend * 100:.1f} cm (cloth {mesh["width"] * 100:.0f} x {mesh["height"] * 100:.0f} cm); air time constant {rho_A / AIR:.2f} s')

In [ ]:
# 고른 값, 상한, 그보다 10배 뻣뻣한 값을 영상 위에: 상한을 넘으면 정말 달라 보이나요?
views = []
for y in (YOUNG, UPPER, UPPER * 10):
    S, X = fit.simulate(obs, MESH, y, AIR); ov = viz.overlay_sim(frames, obs, X, mesh['F'], label=f'young {y:.2g}, air {AIR:.2f}')
    views += [ov[i] for i in np.linspace(0, len(ov) - 1, 4).astype(int)]
    if y == YOUNG: display(Video(viz.write_video(f'{OUT}/fit_overlay.mp4', ov), embed=True, width=640))
viz.show(viz.sheet(views, cols=4), 'rows: chosen young / upper bound / 10x the bound', (18, 10))

**(보고서 ablation) 하나 이상 골라 해 보세요** (시간이 한 번 더 듭니다):
- `space='3d'`: 영상 위 대신 3-D 점으로 비교하면 곡선이 어떻게 바뀌나? (`fit.run_air(obs, MESH, space='3d')`, `fit.run_young(..., space='3d')`)
- `GRAVITY = 'floor'`로 ③부터 다시: 공기 저항과 상한이 얼마나 바뀌나?
- **손 평활 창 7 vs 15** (`HAND_SMOOTH`): ③부터 다시 돌려 손 떨림(`hand_jitter_mm`), 공기 저항, 상한 비교. 창이 너무 크면 진짜 손 움직임까지 지워짐 (⑥ 설명)
- 공기 저항을 young 1e3, 1e5에 고정해서 다시 찾기 (`fit.run_air(obs, MESH, young=1e5)`): 최솟값이 정말 young과 무관한가?
- `CLOTH_MASS_G`를 ±30 % 바꾸면 공기 저항과 상한이 비례해서 바뀌나?

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    A_1e5 = fit.run_air(obs, MESH, young=1e5)                          # 예: young을 바꿔서 공기 저항을 다시
    print(f'air at young 1e3.5: {A["air"]:.2f}   at young 1e5: {A_1e5["air"]:.2f}   ({np.log10(A_1e5["air"] / A["air"]):+.2f} decades)')
    viz.plot_air(A_1e5, 'air sweep at young 1e5')

## 6. 검증: 정답을 아는 합성 영상 (M5)

**왜.** 실제 영상에서는 정답을 모르므로 파이프라인이 맞는지 알 수 없습니다. 그래서 **공기 저항과 강성을 알고 있는** 천을 같은 시뮬레이터로 만들어
③–⑤를 그대로 돌립니다. 통과 기준은 두 가지입니다.
1. 찾은 공기 저항이 정답과 **0.3 decade (2배) 안** (교수님 실험: −0.08, +0.26, +0.29 decade. 깊이 잡음으로 손 궤적이 떨리면 위로 치우침)
2. 정답 강성이 찾은 **상한 아래** (상한이 정답을 "잘라내지" 않음)

**`synth.make_synth(MESH, young=..., air=..., T=10)` 안에서:**
- 정해진 손 움직임(10초: 두 손을 30 ↔ 16 cm로 천천히 모았다 펴고, 조금 옆 · 앞으로, 마지막에 작게 흔듦)으로 천을 시뮬레이션하고 가상 카메라로 렌더링
- **완벽한 추적**(가려진 점은 visibility = False) + **실제 깊이 모델만큼 틀린 깊이**: 프레임마다 1.3–1.7배로 변하는 scale과 매끄러운 3.5 % 잡음
- 합성 장면의 카메라는 수평이고 바닥이 넓게 보이므로 중력은 바닥 평면으로 정합니다 (세로선 없음)

정답 공기 저항을 기본값(0.85)이 아닌 값으로 두어, 파이프라인이 "기본값 근처"를 답하는 것이 아님을 확인합니다.

> **손 평활 창은 손 궤적이 얼마나 떨리는지에 맞춰야 합니다.** 합성 영상의 손은 이미 매끄럽고 마지막에 1.2 s 주기로 흔듭니다.
> 15프레임(0.5 s) 창은 이 흔들림까지 깎아서 공기 저항을 아래로 끌어내립니다 (정답 1.5 → 창 15: 0.49 ✗, 창 7: 1.48 ✓). 그래서 이 셀은 `hand_smooth=7`을 씁니다.
> 실제 영상 T01은 깊이 잡음으로 손 궤적이 떨려서 15가 필요했습니다 (손실 42 → 33 mm). 여러분 영상에서는 7과 15를 둘 다 해 보세요 (ablation).

In [ ]:
TRUE_YOUNG, TRUE_AIR = 2e4, 1.5
Sy = synth.make_synth(MESH, young=TRUE_YOUNG, air=TRUE_AIR, T=10.)
o_s, d_s = lift.lift_take(Sy['tracks'], Sy['depth'], Sy['K'], Sy['reg'], mesh, Sy['t'], hand_smooth=7)   # 합성 손은 이미 매끄러움: 7 (위 설명)
r_s = fit.run_fit(o_s, MESH, verbose=False)                            # ⑤a + ⑤b + ⑤c, 약 10분
air_err = float(np.log10(r_s['air_value'] / TRUE_AIR)); below = TRUE_YOUNG <= r_s['young_upper']
print(f'synthetic: air truth {TRUE_AIR} -> found {r_s["air_value"]:.2f} ({air_err:+.2f} decades)  {"PASS" if abs(air_err) <= .3 else "FAIL"}')
print(f'           young truth {TRUE_YOUNG:.0e} <= upper bound {r_s["young_upper"]:.3g}?  {"PASS" if below else "FAIL"}  (chosen {r_s["young_value"]:.3g})')
viz.plot_fit(r_s, f'synthetic take: truth air {TRUE_AIR}, young {TRUE_YOUNG:.0e}')
display(Video(viz.write_video(f'{OUT}/synthetic.mp4', list(Sy['video'])), embed=True, width=480))

> **생각해 볼 것.** 합성 영상은 손 궤적이 매끄럽고 혼돈이 적어서, 강성 곡선에도 실제 영상보다 뚜렷한 골짜기가 생길 수 있습니다 (교수님 실험: 정답 2e4 근처).
> 그런데도 실제 영상에서 평평하다면 그 차이는 무엇 때문일까요? (손으로 쥐는 방식, 추적 오차, 더 큰 깊이 오차, 손 떨림, 실제 공기 저항은 속도에 비례하지 않음 …)

## 7. 결과 저장 → `results_<팀>.zip`

`results.json`에 찾은 공기 저항, 강성 상한과 고른 값, 두 곡선, 사용한 설정(K, 프레임, 중력, 깊이 모델, 격자)과 합성 검증 결과를 모으고, `obs.npz`(3-D 관찰)와 함께 zip으로 받습니다.
폴더 안: `mesh/` (⓪의 메쉬와 확인 그림), `tracks.npz`, `depth.npz`, `tracks.mp4`, `fit_overlay.mp4`, `synthetic.mp4`, (선택) `masks.mp4`, `results.json`, `obs.npz`.

In [ ]:
res = dict(team=TEAM, version='v7',
           air=AIR, air_fit=A['air'], air_curve=A['curve'], young_ref=A['young_ref'],
           young=YOUNG, young_upper=Y['young_upper'], young_no_bound=Y['no_bound'], young_dr=list(YOUNG_DR),
           loss_floor_mm=Y['floor_mm'], tol_mm=Y['tol_mm'], noise_mm=Y['noise_mm'], young_curve=Y['curve'], space=Y['space'],
           K=np.asarray(K).tolist(), K_source=K_SOURCE, frames=[START, END], fit_end=FIT_END, registration=reg, depth_model=DEPTH_MODEL, grid=GRID,
           gravity=diag['gravity'], up_camera=None if up is None else np.asarray(up).tolist(), tilt_deg=diag['tilt_deg'], hand_smooth=HAND_SMOOTH,
           camera_height=diag['camera_height'], points_seen=diag['seen'],
           cloth=dict(mesh=MESH, width_cm=round(mesh['width'] * 100, 2), height_cm=round(mesh['height'] * 100, 2), mass_g=CLOTH_MASS_G, n_vertices=int(len(mesh['V'])), mesh_report=mrep),
           synthetic=dict(true_air=TRUE_AIR, found_air=r_s['air_value'], air_error_decades=air_err,
                          true_young=TRUE_YOUNG, young_upper=r_s['young_upper'], truth_below_bound=bool(below)))
json.dump(res, open(f'{OUT}/results.json', 'w'), indent=1, default=float); np.savez(f'{OUT}/obs.npz', **{k: v for k, v in obs.items()})
shutil.copytree(MESH, f'{OUT}/mesh', dirs_exist_ok=True)                       # 메쉬도 같이 제출
shutil.make_archive(OUT, 'zip', OUT); print(sorted(os.listdir(OUT)))
if IN_COLAB: files.download(f'{OUT}.zip')